In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd

In [25]:
import importlib
import src.imputation.imputation as imp

importlib.reload(imp)

<module 'src.imputation.imputation' from 'c:\\Users\\lebar0040\\OneDrive - University of Bergen\\ai-analytics-agent\\src\\imputation\\imputation.py'>

In [21]:
import pandas as pd

file_path = project_root / "data" / "interim" / "HCC_proteomics_filtered.csv"

df_filtered = pd.read_csv(file_path)
df_filtered.head()

,idx,111,114,124,126,128,132,136,138,142,...,1022,1026,1028,1032,1042,1044,1046,448,536,698
0,ENSG00000000003.15,25.123487,24.792504,24.813667,24.556260,24.454895,24.808779,25.026977,24.389151,24.473884,...,24.755773,24.407858,24.315442,24.453317,24.396582,24.540703,24.918928,24.876518,24.128921,24.552410
1,ENSG00000000419.12,26.441628,26.253745,26.372726,26.092552,26.482692,26.311869,26.234817,26.126119,26.158555,...,26.125231,26.021210,25.968990,26.313476,25.965280,26.123274,26.109980,26.063115,25.605649,26.076056
2,ENSG00000000457.14,23.319190,23.337303,23.175404,23.405809,23.226412,23.402456,23.529123,23.450332,23.286908,...,23.176737,23.071129,23.280037,23.063194,22.999710,23.304137,23.404143,22.993107,22.830460,22.914256
3,ENSG00000000938.13,21.392391,21.425340,21.522826,21.598335,21.432756,21.201646,21.367521,21.457526,21.352120,...,20.830519,21.417127,21.410856,20.710182,20.860214,21.412469,21.509806,21.002283,21.438820,21.669404
4,ENSG00000000971.16,27.351534,27.249432,27.481766,27.090395,26.593016,26.985530,27.192625,27.126479,27.301768,...,27.251683,27.204168,27.511532,26.356397,26.993061,26.872830,26.918729,27.286199,27.417220,27.125757


In [22]:
df_median = imp.median_imputation(
    df_filtered,
    id_column="idx"
)

In [23]:
import importlib
import src.imputation.imputation as imp

importlib.reload(imp)

df_masked, validation_mask, true_values = imp.create_random_mask(
    df_filtered,
    id_column="idx",
    mask_fraction=0.05,
    random_state=42,
)

In [24]:
print(
    "Artificially masked values:",
    validation_mask.sum().sum()
)

print(
    "Original missing values:",
    df_filtered.drop(columns=["idx"]).isna().sum().sum()
)

print(
    "Missing values after masking:",
    df_masked.drop(columns=["idx"]).isna().sum().sum()
)

Artificially masked values: 67399
Original missing values: 32725
Missing values after masking: 100124


In [26]:
df_median_validation = imp.median_imputation(
    df_masked,
    id_column="idx"
)

In [27]:
median_results = imp.evaluate_imputation(
    imputed_df=df_median_validation,
    true_values=true_values,
    validation_mask=validation_mask,
    id_column="idx",
)

median_results

{'MAE': 0.2068416131249871, 'RMSE': 0.310453648769311, 'n_evaluated': 67399}

In [37]:
import importlib
import src.imputation.imputation as imp

importlib.reload(imp)

<module 'src.imputation.imputation' from 'c:\\Users\\lebar0040\\OneDrive - University of Bergen\\ai-analytics-agent\\src\\imputation\\imputation.py'>

In [29]:
df_knn_validation = imp.knn_imputation(
    df_masked,
    id_column="idx",
    n_neighbors=5,
)

In [30]:
knn_results = imp.evaluate_imputation(
    imputed_df=df_knn_validation,
    true_values=true_values,
    validation_mask=validation_mask,
    id_column="idx",
)

knn_results

{'MAE': 0.15853700695714237, 'RMSE': 0.24170093004629767, 'n_evaluated': 67399}

In [36]:
best_k, knn_tuning_results = imp.select_optimal_knn(
    df_masked=df_masked,
    true_values=true_values,
    validation_mask=validation_mask,
    id_column="idx",
)

print("Automatically selected k:", best_k)

knn_tuning_results

Automatically selected k: 5


,k,MAE,RMSE,n_evaluated
0,3,0.162637,0.247735,67399
1,5,0.158537,0.241701,67399
2,7,0.159459,0.243007,67399
3,9,0.160972,0.245282,67399
4,11,0.162688,0.248063,67399
5,12,0.163535,0.249435,67399


In [38]:
best_k_repeated, knn_summary, knn_detailed = (
    imp.repeated_knn_validation(
        df=df_filtered,
        id_column="idx",
        mask_fraction=0.05,
        random_states=[42, 123, 456, 789, 2026],
    )
)

print(
    "Best k from repeated validation:",
    best_k_repeated
)

knn_summary

Best k from repeated validation: 5


,k,mean_MAE,std_MAE,mean_RMSE,std_RMSE,n_runs
0,5,0.158000,0.000610,0.241678,0.001108,5
1,7,0.158571,0.000604,0.242261,0.001115,5
2,9,0.160182,0.000558,0.244795,0.001070,5
3,11,0.161758,0.000596,0.247363,0.001215,5
4,3,0.162124,0.000565,0.248231,0.001797,5
5,12,0.162582,0.000592,0.248625,0.001157,5


In [39]:
results_path = (
    project_root
    / "data"
    / "interim"
    / "knn_validation_results.csv"
)

knn_summary.to_csv(
    results_path,
    index=False,
)